In [ ]:
"""

Title: "Scalable Neural Safety Certification via Monotonicity"
Code for: Code for The role of the inductive parameter K in sample complexity section of Appendix.
Author: Amirreza Alavi
Contact: seyedamirreza.alavi@colorado.edu
Affiliation: University of Colorado Boulder
Date: 04/2026

"""

In [ ]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers, constraints, initializers
import matplotlib.pyplot as plt
from numpy import linalg as LA
from scipy import stats
from functools import partial
%matplotlib inline
import numpy as np
from scipy.optimize import fsolve
from tensorflow.keras.layers import Input, Dense, Concatenate, Add
from tensorflow.keras import Model
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model
import time
from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D  
from math import inf
import heapq
from dataclasses import dataclass
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection

In [ ]:
# System Definition

def phi(x):
    return tf.constant(10,dtype=tf.float32)*(1-tf.exp(-0.1*tf.constant(x,dtype=tf.float32)))

def sigma(x):
    return  tf.maximum(0,tf.constant(10,dtype=tf.float32)*(1- tf.pow(x/tf.constant(50,dtype=tf.float32),3)))


def f_m(q):
    q = tf.convert_to_tensor(q, dtype=tf.float32)
    tau = tf.constant(0.5, dtype=tf.float32)

    x1 = q[..., 0]
    x2 = q[..., 1]

    y1 = x1 + tau * (8.3 - tf.minimum(phi(x1), sigma(x2)))
    y2 = x2 + tau * (tf.minimum(phi(x1), sigma(x2)) - phi(x2))

    y = tf.stack([y1, y2], axis=-1)
    return tf.clip_by_value(y, 0.0, 50)


def simulate(q0_batch, steps):
    q = tf.convert_to_tensor(q0_batch, tf.float32)
    traj = [q]
    for _ in range(steps):
        q = f_m(q)
        traj.append(q)
    return tf.stack(traj, axis=1).numpy()

In [ ]:
# Gridding Algorithem

@dataclass(order=True)
class _Item:
    
    priority: tuple
    idx: int
    lo: np.ndarray
    hi: np.ndarray

def grid_box(lo, hi, n):
    lo = np.asarray(lo, dtype=float).copy()
    hi = np.asarray(hi, dtype=float).copy()
    assert lo.shape == hi.shape and lo.ndim == 1, "lo, hi must be 1-D of same length"
    assert np.all(hi > lo), "Require hi > lo elementwise"
    assert int(n) == n and n >= 1, "n must be a positive integer"
    n = int(n)

    def vol(l, h):
        return float(np.prod(h - l))

    counter = 0
    heap = []
    heapq.heappush(heap, _Item(priority=(-vol(lo, hi), counter), idx=0, lo=lo, hi=hi))
    counter += 1

    boxes = []
    while len(heap) + len(boxes) < n:
        item = heapq.heappop(heap)
        l, h = item.lo, item.hi
        lengths = h - l
        axis = int(np.argmax(lengths))  
        mid = 0.5 * (l[axis] + h[axis])

        l1, h1 = l.copy(), h.copy()
        l2, h2 = l.copy(), h.copy()
        h1[axis] = mid
        l2[axis] = mid

        heapq.heappush(heap, _Item(priority=(-vol(l1, h1), counter), idx=counter, lo=l1, hi=h1))
        counter += 1
        heapq.heappush(heap, _Item(priority=(-vol(l2, h2), counter), idx=counter, lo=l2, hi=h2))
        counter += 1

    
    while heap:
        boxes.append(heapq.heappop(heap))

    
    boxes.sort(key=lambda x: x.idx)
    lows  = np.stack([b.lo for b in boxes], axis=0)
    highs = np.stack([b.hi for b in boxes], axis=0)
    return lows, highs


In [ ]:
# Sampled points for a dense gridding

N=2
xi_und, xi_bar = grid_box([0.0, 0.0], [40.0, 40.0], n=100000)
xin_und, xin_bar = grid_box([0.0, 0.0], [10.0, 10.0], n=1000)
xuns1_und, xuns1_bar = grid_box([40.0, 0.0], [50.0, 50.0], n=1000)
xuns2_und, xuns2_bar = grid_box([0.0, 40.0], [40.0, 50.0], n=1000)
xuns_und = np.vstack([xuns1_und, xuns2_und])
xuns_bar = np.vstack([xuns1_bar, xuns2_bar])

In [ ]:
# Barrier Neural Network Definition

max_layer = layers.Lambda(lambda x: tf.reduce_max(x, axis=1, keepdims=True))

modelB = tf.keras.Sequential()
modelB.add(layers.Input(shape=(N,)))
modelB.add(max_layer) 
modelB.add(layers.Dense(200, use_bias=True, activation=tf.keras.layers.LeakyReLU(alpha=0.1),
                        kernel_constraint=tf.keras.constraints.NonNeg()))
modelB.add(layers.Dense(100, use_bias=True, activation=tf.keras.layers.LeakyReLU(alpha=0.1),
                        kernel_constraint=tf.keras.constraints.NonNeg()))
modelB.add(layers.Dense(1, use_bias=True, activation=None,
                        kernel_constraint=tf.keras.constraints.NonNeg()))

In [ ]:
# Training Ordinary Monotone Neural Barrier Certificate

epochs = 100000
opt1 = tf.keras.optimizers.Adam(1e-2)
t1 = time.perf_counter()

gamma = 5
eta = 10


for i in range(epochs):
    with tf.GradientTape() as tape0, tf.GradientTape() as tape1:

        loss_B = tf.reduce_sum(tf.nn.relu(tf.subtract(1.0,1.0)))

        loss_B += tf.reduce_sum(tf.square(tf.nn.relu(modelB(f_m(xin_bar))- gamma)))
        loss_B += tf.reduce_sum(tf.square(tf.nn.relu(eta-modelB(xuns_und))))
        loss_B += 10*tf.reduce_sum(tf.math.sign(tf.nn.relu(-(modelB(xi_und) - 9.5)))  * (tf.nn.relu(modelB(f_m(xi_bar)) -  9.5)))

                                
        grad_B = tape0.gradient(loss_B, modelB.trainable_variables)
                
        if loss_B == 0   and i>1:
            print('Safety Verified')
            break
            
        if loss_B!=0:
            opt1.apply_gradients(zip(grad_B, modelB.trainable_variables),)
            
        if i%100 == 1:
            print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_sum(tf.square(tf.nn.relu(modelB(f_m(xin_bar))- gamma))).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_sum(tf.square(tf.nn.relu(eta-modelB(xuns_und)))).numpy():.4f}, "
            f"Third condition = {tf.reduce_sum(tf.math.sign(tf.nn.relu(-(modelB(xi_und) - 9.5)))  * (tf.nn.relu(modelB(f_m(xi_bar)) -  9.5))).numpy():.4f}, "
            f"LossB = {loss_B.numpy():.4f}"
            )


    
t2 = time.perf_counter()
tot = t2 - t1
print(f'Total runtime: {tot}s')
print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_sum(tf.square(tf.nn.relu(modelB(f_m(xin_bar))- gamma))).numpy():.4f}, "
            f"minB[xunsafe_und] = {tf.reduce_sum(tf.square(tf.nn.relu(eta-modelB(xuns_und)))).numpy():.4f}, "
            f"Third condition = {tf.reduce_sum(tf.math.sign(tf.nn.relu(-(modelB(xi_und) - 9.5)))  * (tf.nn.relu(modelB(f_m(xi_bar)) -  9.5))).numpy():.4f}, "
            f"LossB = {loss_B.numpy():.4f}"
            )

In [ ]:
# Sub-level set of ordinary barrier on a dense gridding 

def rectangles_to_linecollection(lows, highs, *, lw=0.25, alpha=0.35, zorder=4, rasterized=True):

    lows = np.asarray(lows, dtype=np.float32)
    highs = np.asarray(highs, dtype=np.float32)

    x0 = lows[:, 0]; y0 = lows[:, 1]
    x1 = highs[:, 0]; y1 = highs[:, 1]

    segs = np.empty((4 * lows.shape[0], 2, 2), dtype=np.float32)

    segs[0::4, 0, 0] = x0; segs[0::4, 0, 1] = y0
    segs[0::4, 1, 0] = x1; segs[0::4, 1, 1] = y0

    segs[1::4, 0, 0] = x1; segs[1::4, 0, 1] = y0
    segs[1::4, 1, 0] = x1; segs[1::4, 1, 1] = y1

    segs[2::4, 0, 0] = x1; segs[2::4, 0, 1] = y1
    segs[2::4, 1, 0] = x0; segs[2::4, 1, 1] = y1

    segs[3::4, 0, 0] = x0; segs[3::4, 0, 1] = y1
    segs[3::4, 1, 0] = x0; segs[3::4, 1, 1] = y0

    lc = LineCollection(segs, linewidths=lw, alpha=alpha, zorder=zorder)
    lc.set_rasterized(rasterized)  
    return lc


delta = 9.5

x = np.linspace(0, 50, 300)
y = np.linspace(0, 50, 300)
X1, X2 = np.meshgrid(x, y)
grid = np.stack([X1, X2], axis=-1).reshape(-1, 2).astype(np.float32)

B = modelB(tf.constant(grid, dtype=tf.float32))
B = tf.reshape(B, (-1,))
B_map = B.numpy().reshape(X1.shape)


fig, ax = plt.subplots(figsize=(6, 6))


ax.contourf(X1, X2, B_map, levels=[-np.inf, delta], colors=["lightgray"], alpha=0.5, zorder=1)
ax.contour(X1, X2, B_map, levels=[delta], colors=["black"], linewidths=1.4, zorder=2)

ax.add_collection(rectangles_to_linecollection(xi_und, xi_bar, lw=0.20, alpha=0.3, zorder=3, rasterized=True))


ax.add_patch(mpatches.Rectangle((0, 0), 10, 10, facecolor="blue", edgecolor="none", alpha=0.4, zorder=5))
ax.add_patch(mpatches.Rectangle((40, 0), 10, 50, facecolor="red", edgecolor="none", alpha=0.4, zorder=5))
ax.add_patch(mpatches.Rectangle((0, 40), 40, 10, facecolor="red", edgecolor="none", alpha=0.4, zorder=5))


ax.add_patch(mpatches.Rectangle((0, 0), 50, 50, fill=False, edgecolor="cyan", linewidth=0.5, alpha=0.6, zorder=6))
ax.add_collection(rectangles_to_linecollection(xin_und, xin_bar, lw=0.35, alpha=0.8, zorder=6, rasterized=True))
ax.add_collection(rectangles_to_linecollection(xuns1_und, xuns1_bar, lw=0.35, alpha=0.8, zorder=6, rasterized=True))
ax.add_collection(rectangles_to_linecollection(xuns2_und, xuns2_bar, lw=0.35, alpha=0.8, zorder=6, rasterized=True))


ax.set_xlim(0, 50); ax.set_ylim(0, 50)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(r"$x_1$", fontsize=16)
ax.set_ylabel(r"$x_2$", fontsize=16)
ax.set_xticks(np.arange(0, 51, 10))
ax.set_yticks(np.arange(0, 51, 10))

init_patch   = mpatches.Patch(color="blue", alpha=0.4, label=r"Initial set ($\mathcal{X}_0$)")
unsafe_patch = mpatches.Patch(color="red",  alpha=0.4, label=r"Unsafe set ($\mathcal{X}_u$)")
sub_patch    = mpatches.Patch(color="lightgray", alpha=0.3, label=r"$B(x)\le \delta$")
grid_line    = mpatches.Patch(facecolor="none", edgecolor="cyan", label=r"Hyper-rectangular gridding", alpha=0.5)

ax.legend(handles=[init_patch, unsafe_patch, sub_patch, grid_line], loc="best", frameon=True, fontsize=12)

plt.tight_layout()
#plt.savefig("fullgrid.pdf", bbox_inches="tight")
plt.show()
